Stage1



In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.5/334.5 kB 15.4 MB/s eta 0:00:00


Hugging Face login

In [ ]:
from huggingface_hub import login
login()


Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


Load Gemma model (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "google/gemma-2b-it"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 🔴 Correct 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

GemmaForCausalLM(
  (model): GemmaModel(
    (embed_tokens): Embedding(256000, 2048, padding_idx=0)
    (layers): ModuleList(
      (0-17): 18 x GemmaDecoderLayer(
        (self_attn): GemmaAttention(
          (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): GemmaMLP(
          (gate_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (up_proj): Linear4bit(in_features=2048, out_features=16384, bias=False)
          (down_proj): Linear4bit(in_features=16384, out_features=2048, bias=False)
          (act_fn): GELUActivation()
        )
        (input_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): GemmaRMSNorm((2048,), eps=1e-06)
      )
    )
    (n

In [ ]:
!pip install -q bitsandbytes accelerate transformers peft

— Upload math book PDF

In [ ]:
from google.colab import files

uploaded = files.upload()  # Upload math_book.pdf

Convert PDF → text

In [ ]:
from pypdf import PdfReader

reader = PdfReader("math_book.pdf")

raw_text = []
for page in reader.pages:
    text = page.extract_text()
    if text:
        raw_text.append(text)

full_text = "\n".join(raw_text)

print("PDF extracted")

In [ ]:
clean_lines = []

for line in full_text.split("\n"):
    line = line.strip()

    if len(line) < 40:
        continue
    if "Answer" in line or "Solution" in line:
        continue
    if any(char.isdigit() for char in line) and "=" in line:
        continue

    clean_lines.append(line)

print("Clean lines:", len(clean_lines))

Create dataset

In [ ]:
from datasets import Dataset

dataset = Dataset.from_dict({
    "text": clean_lines
})

dataset

Tokenize dataset

In [ ]:
def tokenize(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # \ THIS LINE FIXES YOUR ERROR
    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_ds = dataset.map(
    tokenize,
    batched=True,
    remove_columns=["text"]
)

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

#  VERY IMPORTANT STEP

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # works for Gemma
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

Training configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-math-domain",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=200,
    save_total_limit=1,
    report_to="none"
)

Train (Math Book Adaptation)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds
)

trainer.train()

In [ ]:
model.save_pretrained("gemma-math-domain-lora")
tokenizer.save_pretrained("gemma-math-domain-lora")

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("gemma-math-domain-lora", "zip", "gemma-math-domain-lora")
files.download("gemma-math-domain-lora.zip")
print("Download started!")

Light sanity check

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# 🔹 Base model
model_name = "google/gemma-2b"

# 🔹 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 🔹 4-bit quantization config (correct way for Gemma)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 🔹 Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# 🔹 Load LOCAL LoRA adapters (FIXED PATH)
model = PeftModel.from_pretrained(
    model,
    "./gemma-math-domain-lora"
)

# 🔹 Merge adapters into base model (recommended for inference)
model = model.merge_and_unload()

model.eval()

# 🔹 Sanity test prompt
prompt = "Explain what a mathematical function is."

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# 🔹 Generate output
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

# 🔹 Print result
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
import gc
import torch

try:
    del model
except:
    pass

try:
    del tokenizer
except:
    pass

torch.cuda.empty_cache()
gc.collect()

print("Cleanup completed")

In [ ]:
# Enable training for LoRA
model.enable_input_require_grads()
model.print_trainable_parameters()

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile("gemma-math-domain-lora.zip", 'r') as zip_ref:
    zip_ref.extractall("gemma-math-domain-lora")

In [ ]:
## Load Stage-1 LoRA adapters
model = PeftModel.from_pretrained(
    model,
   "./gemma-math-domain-lora"
)

# 🔴 ADD THIS PART (Stage-2 LoRA)
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

# Check trainable params
model.print_trainable_parameters()

# Train mode
model.train()

 **Stage 2
**


Generate base dataset:

Load into HuggingFace Dataset

**----------------------------------------------------------------------------------------------------------------------------------------------------------------**

Domain specific fine tuning:

Install and extract from both PDFs:

In [ ]:
!pip install pypdf -q

from pypdf import PdfReader
import re, json, random

# Extract math book text
math_reader = PdfReader("math_book.pdf")
math_text = ""
for page in math_reader.pages:
    math_text += page.extract_text() + "\n"

# Extract history book text
hist_reader = PdfReader("History-Class-7th.pdf")
hist_text = ""
for page in hist_reader.pages:
    hist_text += page.extract_text() + "\n"

print("Math text length:", len(math_text))
print("History text length:", len(hist_text))

Extract real questions from both books:

In [ ]:
def extract_math_questions(text):
    lines = text.split('\n')
    questions = []
    for line in lines:
        line = line.strip()
        # Numbered exercise questions with enough content
        if re.match(r'^\d+\.', line) and len(line) > 25 and len(line) < 250:
            # Skip chapter headings and section titles
            if not any(skip in line for skip in ["Chapter", "EXERCISE", "Fig", "NCERT"]):
                questions.append(line)
    return list(set(questions))  # remove duplicates

def extract_history_questions(text):
    lines = text.split('\n')
    questions = []
    for line in lines:
        line = line.strip()
        if '?' in line and len(line) > 25 and len(line) < 250:
            # Only keep lines that start with question words or numbers
            if re.match(r'^(\d+\.|What|Who|How|Why|When|Where|Describe|Explain|Discuss|Analyse)', line):
                questions.append(line)
    return list(set(questions))

math_questions = extract_math_questions(math_text)
history_questions = extract_history_questions(hist_text)

print(f"Math questions extracted: {len(math_questions)}")
print(f"History questions extracted: {len(history_questions)}")
print("\nSample math:")
for q in math_questions[:5]:
    print(" ", q)
print("\nSample history:")
for q in history_questions[:5]:
    print(" ", q)

Build harder/easier pairs from real textbook questions:

In [ ]:
# Math harder/easier transformation rules
def make_math_harder(q):
    # Add a constraint or extra operation
    templates = [
        f"Solve the following and verify your answer: {q}",
        f"{q} Also find the value if the result is doubled.",
        f"{q} Express your answer as a fraction and a decimal.",
    ]
    return random.choice(templates)

def make_math_easier(q):
    # Simplify by breaking into a hint
    templates = [
        f"Hint: Start by identifying the numbers. {q}",
        f"Fill in the blank to solve: {q}",
        f"Use a number line to solve: {q}",
    ]
    return random.choice(templates)

def make_history_harder(q):
    q_clean = q.lstrip("0123456789. ").strip()
    templates = [
        f"Critically analyze: {q_clean} Support your answer with examples.",
        f"With reference to medieval India, {q_clean.lower()} Discuss in detail.",
        f"Examine the causes and effects: {q_clean}",
    ]
    return random.choice(templates)

def make_history_easier(q):
    q_clean = q.lstrip("0123456789. ").strip()
    # Strip it to the core subject
    words = q_clean.split()
    subject = " ".join(words[:6])
    templates = [
        f"In one sentence, answer: {q_clean}",
        f"Name one example of: {subject}",
        f"True or False: {subject}",
    ]
    return random.choice(templates)

# Build dataset
dataset = []

# Math pairs
for q in math_questions:
    # Harder
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": q,
        "output": make_math_harder(q),
        "direction": "harder",
        "subject": "math",
        "concept": "textbook_exercise"
    })
    # Easier
    dataset.append({
        "instruction": "Rewrite the question to easier difficulty while keeping the same concept.",
        "input": q,
        "output": make_math_easier(q),
        "direction": "easier",
        "subject": "math",
        "concept": "textbook_exercise"
    })

# History pairs
for q in history_questions:
    # Harder
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": q,
        "output": make_history_harder(q),
        "direction": "harder",
        "subject": "history",
        "concept": "textbook_conceptual"
    })
    # Easier
    dataset.append({
        "instruction": "Rewrite the question to easier difficulty while keeping the same concept.",
        "input": q,
        "output": make_history_easier(q),
        "direction": "easier",
        "subject": "history",
        "concept": "textbook_conceptual"
    })

random.shuffle(dataset)
print(f"Total dataset size: {len(dataset)}")
print(f"Math harder: {sum(1 for d in dataset if d['subject']=='math' and d['direction']=='harder')}")
print(f"Math easier: {sum(1 for d in dataset if d['subject']=='math' and d['direction']=='easier')}")
print(f"History harder: {sum(1 for d in dataset if d['subject']=='history' and d['direction']=='harder')}")
print(f"History easier: {sum(1 for d in dataset if d['subject']=='history' and d['direction']=='easier')}")

print("\nSample harder math:")
print(next(d for d in dataset if d['subject']=='math' and d['direction']=='harder'))
print("\nSample easier history:")
print(next(d for d in dataset if d['subject']=='history' and d['direction']=='easier'))

Generate and download new 8000 dataset:

In [ ]:
import json, random
from google.colab import files

dataset = []

names = ["Sara","Aman","Priya","Rahul","Arjun","Maya","John","Riya"]
objects = ["books","pens","marbles","apples","stickers","notebooks"]

def generate_linear():
    a = random.randint(2,8)
    b = random.randint(1,9)
    c = random.randint(10,40)
    base = f"Solve {a}x + {b} = {c}"
    harder = f"Solve {a}(2x - {b}) + {b+3} = {2*c - b + 3}"
    return base, harder

def generate_counting():
    name = random.choice(names)
    obj = random.choice(objects)
    n1 = random.randint(3,15)
    n2 = random.randint(2,10)
    base = f"{name} has {n1} {obj}. {name} buys {n2} more. How many {obj} does {name} have now?"
    harder = f"{name} has {n1} {obj}. {name} buys {n2} more, gives 3 away, and later buys 4 more. How many {obj} does {name} finally have?"
    return base, harder

history = [
    ("Who was Akbar?",
     "Explain the major administrative reforms introduced by Akbar during the Mughal Empire."),
    ("What was the Delhi Sultanate?",
     "Describe the origin and political significance of the Delhi Sultanate in medieval India."),
    ("What was the Bhakti movement?",
     "Analyze the social and religious impact of the Bhakti movement in medieval India."),
    ("What were medieval Indian trading towns?",
     "Explain the economic importance of trading towns in medieval India."),
    ("What is a monument?",
     "Discuss how monuments help historians understand medieval Indian architecture.")
]

# Generate base harder examples
for _ in range(1500):
    inp, out = generate_linear()
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": inp, "output": out,
        "subject": "math", "concept": "linear_equation"
    })

for _ in range(1500):
    inp, out = generate_counting()
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": inp, "output": out,
        "subject": "math", "concept": "counting_problem"
    })

for _ in range(1000):
    q = random.choice(history)
    dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": q[0], "output": q[1],
        "subject": "history", "concept": "conceptual"
    })

# Now double it by adding easier examples (swap input/output)
full_dataset = []
for example in dataset:
    # Harder entry
    full_dataset.append({
        "instruction": "Rewrite the question to harder difficulty while keeping the same concept.",
        "input": example["input"],
        "output": example["output"],
        "direction": "harder",
        "subject": example["subject"],
        "concept": example["concept"]
    })
    # Easier entry (swap)
    full_dataset.append({
        "instruction": "Rewrite the question to easier difficulty while keeping the same concept.",
        "input": example["output"],
        "output": example["input"],
        "direction": "easier",
        "subject": example["subject"],
        "concept": example["concept"]
    })

random.shuffle(full_dataset)

with open("difficulty_transformation_dataset_8000.json", "w") as f:
    json.dump(full_dataset, f, indent=2)

print("Dataset saved!")
print("Total samples:", len(full_dataset))
print("Harder:", sum(1 for d in full_dataset if d["direction"] == "harder"))
print("Easier:", sum(1 for d in full_dataset if d["direction"] == "easier"))

files.download("difficulty_transformation_dataset_8000.json")

Combine with original synthetic dataset and save:

In [ ]:
# Load original 8000 dataset
with open("difficulty_transformation_dataset_8000.json") as f:
    original_data = json.load(f)

# Combine both
combined = original_data + dataset
random.shuffle(combined)

with open("difficulty_dataset_combined.json", "w") as f:
    json.dump(combined, f, indent=2)

print("Combined dataset saved!")
print("Original samples:", len(original_data))
print("Textbook samples:", len(dataset))
print("Total combined:", len(combined))

Download

In [ ]:
from google.colab import files
files.download("difficulty_dataset_combined.json")

Upload the downloaded dataset back to Colab:

In [ ]:
from google.colab import files
import json

uploaded = files.upload()

with open("difficulty_transformation_dataset_8000.json") as f:
    data = json.load(f)

print("Loaded samples:", len(data))
print("Sample entry:")
print(data[0])

 Load into HuggingFace Dataset:

In [ ]:
from datasets import Dataset

dataset = Dataset.from_list(data)
print(dataset)

Format dataset:

In [ ]:
def format_example(example):

    text = f"""{example['instruction']}

Question:
{example['input']}

Rewritten Question:
{example['output']}"""

    return {"text": text}

formatted_dataset = dataset.map(format_example)

formatted_dataset = formatted_dataset.remove_columns(
    ["instruction","input","output","subject","concept"]
)

print(formatted_dataset[0])

Reload tokenizer

In [ ]:
from transformers import AutoTokenizer

model_name = "google/gemma-2b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

Tokenize

In [ ]:
tokenized_dataset = formatted_dataset.map(tokenize)

 Tokenization:

In [ ]:
def tokenize(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=256
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

tokenized_dataset = formatted_dataset.map(tokenize)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])

print(tokenized_dataset)

Training config:

In [ ]:
## Clearing GPU
import torch, gc

torch.cuda.empty_cache()
gc.collect()

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./gemma-stage2",

    num_train_epochs=3,   # reduce from 5 → 3 (major time save)

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,  # reduce from 8 → faster training

    learning_rate=2e-4,

    logging_steps=20,
    save_strategy="epoch",

    fp16=True,
    optim="paged_adamw_8bit",

    report_to="none"
)

Trainer and train:

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

In [ ]:
import torch, gc

torch.cuda.empty_cache()
gc.collect()

trainer.train()

In [ ]:
model = PeftModel.from_pretrained(
    model,
   "./gemma-math-domain-lora"
)

# IMPORTANT FIX
model.enable_input_require_grads()
model.print_trainable_parameters()

model.train()

Save Trained Model:

In [ ]:
model.save_pretrained("difficulty_model_stage2_final")
tokenizer.save_pretrained("difficulty_model_stage2_final")
print("Model saved!")

 Zip and download trained model:

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("difficulty_model_stage2_final", "zip", "difficulty_model_stage2_final")
files.download("difficulty_model_stage2_final.zip")
print("Download started!")

Import Libraries

Part 2 ---------------------------------------------------------------------------------------------------------------------------------


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets

Login to HuggingFace

In [ ]:
from huggingface_hub import login
login()

Load the base Gemma model (4-bit)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_name = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

In [ ]:
!pip uninstall -y sympy
!pip install sympy

Import saved trained model

In [ ]:
from google.colab import files
files.upload()

Unzip

In [ ]:
import zipfile

with zipfile.ZipFile("stage2_gemma_lora.zip", 'r') as zip_ref:
    zip_ref.extractall("stage2_gemma_lora")

load stage1 LoRA adapters

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

model_name = "google/gemma-2b"

# 🔹 Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 🔹 4-bit config (correct for Gemma)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# 🔹 Load base model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

# 🔹 Load Stage-1 LoRA adapters (IMPORTANT)
model = PeftModel.from_pretrained(
    model,
    "./gemma-math-domain-lora"
)

# 🔹 Keep in training mode (for Stage 2)
model.train()

Load Trained LoRA

In [ ]:
from peft import PeftModel

model = PeftModel.from_pretrained(
    base_model,
    "stage2_gemma_lora"
)

model.eval()

Testing Prompts

In [ ]:
import re

##Unzip Stage 1 LoRA

In [ ]:
from google.colab import files

# Upload both zips
files.upload()  # upload stage2_gemma_lora.zip and gemma-math-domain-lora.zip

if not STAGE1_DIR.exists():
    with zipfile.ZipFile(STAGE1_ZIP, "r") as z:
        z.extractall(STAGE1_DIR)
    print(f"Unzipped to {STAGE1_DIR}")
else:
    print(f"Already exists: {STAGE1_DIR}")

print("Contents:", [p.name for p in STAGE1_DIR.iterdir()])

##Load base model + Stage 1 LoRA

In [ ]:
from huggingface_hub import login
login()

In [ ]:
def load_base_model():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb,
        device_map="auto",
    )
    return base, tokenizer

clear_memory()
base_model, tokenizer = load_base_model()

# Load stage1 adapter on top
stage1_model = PeftModel.from_pretrained(base_model, str(STAGE1_DIR))
stage1_model = prepare_model_for_kbit_training(stage1_model)
stage1_model.gradient_checkpointing_enable()
stage1_model.config.use_cache = False

print("Stage 1 model loaded.")
stage1_model.print_trainable_parameters()

##Define new training examples

In [ ]:
from google.colab import files

# Upload train.json
files.upload()

with open("train.json") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data)} training examples")

# Build NEW_EXAMPLES from train.json
NEW_EXAMPLES = []
for entry in train_data:
    # Use 'harder' key — fall back to 'medium' if harder not present
    harder_q = entry.get("harder") or entry.get("medium")
    easier_q = entry.get("easy")

    if not harder_q or not easier_q:
        continue  # skip incomplete entries

    NEW_EXAMPLES.append({
        "topic": entry["topic"],
        "base_question": entry["base_question"],
        "harder": harder_q,
        "easier": easier_q,
    })

print(f"Valid training pairs: {len(NEW_EXAMPLES)}")

# Show breakdown by topic
from collections import Counter
topics = Counter(e["topic"] for e in NEW_EXAMPLES)
print("Topics:", dict(topics))
print("\nSample entry:")
print(NEW_EXAMPLES[0])

##Format and tokenize with prompt masking

In [ ]:
def build_prompt_and_full_text(example, difficulty):
    """Returns (prompt_only, prompt+response) for one difficulty direction."""
    prompt = (
        f"### System:\n{SYSTEM_PROMPT}\n\n"
        f"### Input:\n"
        f"Topic: {example['topic']}\n"
        f"Original Question: {example['base_question']}\n"
        f"Target Difficulty: {difficulty}\n\n"
        f"### Response:\n"
    )
    full_text = prompt + example[difficulty]
    return prompt, full_text


def make_instruction_records(examples):
    records = []
    for ex in examples:
        for difficulty in ("harder", "easier"):
            prompt, full_text = build_prompt_and_full_text(ex, difficulty)
            records.append({"prompt": prompt, "full_text": full_text})
    return records


def tokenize_with_masking(example):
    """Mask prompt tokens so the model only trains to generate the answer."""
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )["input_ids"]

    encoded = tokenizer(
        example["full_text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

    labels = encoded["input_ids"].copy()
    prompt_len = min(len(prompt_ids), MAX_LEN)
    # Mask prompt tokens — model does NOT get credit for predicting them
    labels[:prompt_len] = [-100] * prompt_len
    # Mask padding tokens too
    labels = [
        t if m == 1 else -100
        for t, m in zip(labels, encoded["attention_mask"])
    ]
    encoded["labels"] = labels
    return encoded


records = make_instruction_records(NEW_EXAMPLES)
raw_ds  = Dataset.from_list(records)

tokenized_ds = raw_ds.map(
    tokenize_with_masking,
    remove_columns=raw_ds.column_names,
)

print(f"Tokenized dataset: {tokenized_ds}")
print(f"Example — non-masked label tokens: "
      f"{sum(1 for l in tokenized_ds[0]['labels'] if l != -100)}")

##Add Stage 2 LoRA and train

In [ ]:
clear_memory()

# Add a fresh Stage 2 LoRA on top of the Stage 1 model
lora2_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

stage2_model = get_peft_model(stage1_model, lora2_config)
stage2_model.print_trainable_parameters()
stage2_model.train()

stage2_args = TrainingArguments(
    output_dir="./stage2_runs",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=1e-4,
    num_train_epochs=6,          # increase if you add more examples
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

trainer = Trainer(
    model=stage2_model,
    args=stage2_args,
    train_dataset=tokenized_ds,
)

trainer.train()
print("Stage 2 training complete.")

##Save Stage 2 adapter

In [ ]:
STAGE2_ADAPTER.mkdir(parents=True, exist_ok=True)
stage2_model.save_pretrained(str(STAGE2_ADAPTER))
tokenizer.save_pretrained(str(STAGE2_ADAPTER))
print("Saved to:", STAGE2_ADAPTER)
print("Files:", [p.name for p in STAGE2_ADAPTER.iterdir()])

In [ ]:
files.download("stage2_adapter")

##Load inference model (no rules, pure model)

In [ ]:
# ── Pure model inference — zero rule-based logic ──────────────────────────
clear_memory()

infer_base, tokenizer = load_base_model()
infer_model = PeftModel.from_pretrained(infer_base, str(STAGE2_ADAPTER))
infer_model.eval()
print("Inference model ready.")


def generate_rewrite(question: str, topic: str, difficulty: str,
                     max_new_tokens: int = 80) -> str:
    """
    Rewrites `question` to `difficulty` ('harder' or 'easier').
    Pure model generation — no string manipulation, no templates.
    """
    prompt, _ = build_prompt_and_full_text(
        {"topic": topic, "base_question": question, difficulty: ""},
        difficulty,
    )
    # Remove the empty answer from full_text — we only need the prompt
    prompt = (
        f"### System:\n{SYSTEM_PROMPT}\n\n"
        f"### Input:\n"
        f"Topic: {topic}\n"
        f"Original Question: {question}\n"
        f"Target Difficulty: {difficulty}\n\n"
        f"### Response:\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(infer_model.device)
    with torch.no_grad():
        output = infer_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens (strip the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[-1]:]
    result = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Remove any accidental "### ..." continuation
    result = re.split(r"###|Input:|System:", result)[0].strip()
    return result

##Test Evaluation

##Optional: zip and download the Stage 2 adapter

In [ ]:
import shutil
shutil.make_archive("stage2_adapter_final", "zip", str(STAGE2_ADAPTER))
files.download("stage2_adapter_final.zip")
print("Download started.")

NameError: name 'STAGE2_ADAPTER' is not defined

--------------------------------------------------------------------------------------------------------------------------------------------------------------********************************************************************************

In [ ]:
!pip install -U "bitsandbytes>=0.46.1" "accelerate>=0.27.0" "transformers>=4.38.0" peft sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 37.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
import bitsandbytes as bnb
import accelerate
import transformers

print("bitsandbytes:", bnb.__version__)
print("accelerate:", accelerate.__version__)
print("transformers:", transformers.__version__)


bitsandbytes: 0.49.2
accelerate: 1.13.0
transformers: 5.0.0


In [ ]:
import gc, json, random, re, zipfile
from pathlib import Path

import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    BitsAndBytesConfig, Trainer, TrainingArguments,
    set_seed,
)
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training

set_seed(42)
random.seed(42)

BASE_MODEL     = "google/gemma-2b"
STAGE1_ZIP     = "gemma-math-domain-lora.zip"
STAGE1_DIR     = Path("stage1_adapter")
STAGE2_ZIP     = "stage2_gemma_lora.zip"
STAGE2_DIR     = Path("stage2_adapter")
STAGE3_ADAPTER = Path("stage3_adapter")
MAX_LEN        = 384

SYSTEM_PROMPT = (
    "You are a helpful curriculum assistant. "
    "Rewrite the given question to the requested difficulty level "
    "while keeping the topic and subject matter identical. "
    "Return only the rewritten question, nothing else."
)

def clear_memory():
    gc.collect()
    torch.cuda.empty_cache()

print("Setup complete.")

StrictDataclassDefinitionError: Class 'BloomConfig' must be a dataclass before applying @strict.

Cell upload for stage 1 adapters

In [ ]:
from google.colab import files
import shutil
import zipfile

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

stage1_uploaded = next((name for name in uploaded if "gemma-math-domain-lora" in name), None)
if stage1_uploaded is None:
    raise FileNotFoundError("Please upload the Stage 1 zip: gemma-math-domain-lora.zip")

if STAGE1_DIR.exists():
    shutil.rmtree(STAGE1_DIR)

STAGE1_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(stage1_uploaded, "r") as z:
    z.extractall(STAGE1_DIR)

print(f"Stage 1 unzipped to {STAGE1_DIR}")
print("Stage 1 contents:", [p.name for p in STAGE1_DIR.iterdir()])


Saving gemma-math-domain-lora.zip to gemma-math-domain-lora (1).zip
Uploaded files: ['gemma-math-domain-lora (1).zip']
Stage 1 unzipped to stage1_adapter
Stage 1 contents: ['adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'adapter_config.json', 'README.md', 'tokenizer_config.json']


Cell for Stage 2

In [ ]:
from google.colab import files
import shutil
import zipfile

uploaded = files.upload()
print("Uploaded files:", list(uploaded.keys()))

stage2_uploaded = next((name for name in uploaded if "stage2_gemma_lora" in name), None)
if stage2_uploaded is None:
    raise FileNotFoundError("Please upload the Stage 2 zip: stage2_gemma_lora.zip")

if STAGE2_DIR.exists():
    shutil.rmtree(STAGE2_DIR)

STAGE2_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(stage2_uploaded, "r") as z:
    z.extractall(STAGE2_DIR)

print(f"Stage 2 unzipped to {STAGE2_DIR}")
print("Stage 2 contents:", [p.name for p in STAGE2_DIR.iterdir()])


Saving stage2_gemma_lora.zip to stage2_gemma_lora (1).zip
Uploaded files: ['stage2_gemma_lora (1).zip']
Stage 2 unzipped to stage2_adapter
Stage 2 contents: ['adapter_model.safetensors', 'tokenizer.json', 'adapter_config.json', 'README.md', 'tokenizer_config.json']


Load base model + Stage 1 + Stage 2 (replace entirely):

In [ ]:
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel, prepare_model_for_kbit_training

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

login(token=HF_TOKEN)

def load_base_model():
    bnb = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
    tokenizer.pad_token = tokenizer.eos_token

    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        token=HF_TOKEN,
        quantization_config=bnb,
        device_map="auto",
    )
    return base, tokenizer


ithun changes kelet : trainingpn thodi badallli ahe , bas 1 subject specific

In [ ]:
# from google.colab import files
# files.upload()  # upload train.json

# with open("train.json") as f:
#     train_data = json.load(f)

# print(f"Loaded {len(train_data)} training examples")

# # Build NEW_EXAMPLES from correct keys
# NEW_EXAMPLES = []
# for entry in train_data:
#     instruction = entry.get("instruction", "")
#     inp = entry.get("input", "")
#     out = entry.get("output", "")
#     direction = entry.get("direction", "")
#     subject = entry.get("subject", "")

#     if not inp or not out or not direction:
#         continue

#     # Map direction to harder/easier keys
#     if direction == "harder":
#         NEW_EXAMPLES.append({
#             "topic": subject,
#             "base_question": inp,
#             "harder": out,
#             "easier": "",   # will be filled from paired easier entry
#             "direction": "harder",
#             "instruction": instruction,
#         })
#     elif direction == "easier":
#         NEW_EXAMPLES.append({
#             "topic": subject,
#             "base_question": inp,
#             "harder": "",
#             "easier": out,
#             "direction": "easier",
#             "instruction": instruction,
#         })

# print(f"Valid training entries: {len(NEW_EXAMPLES)}")

# from collections import Counter
# directions = Counter(e["direction"] for e in NEW_EXAMPLES)
# subjects = Counter(e["topic"] for e in NEW_EXAMPLES)
# print("Directions:", dict(directions))
# print("Subjects:", dict(subjects))
# print("\nSample harder entry:")
# print(next(e for e in NEW_EXAMPLES if e["direction"] == "harder"))
# print("\nSample easier entry:")
# print(next(e for e in NEW_EXAMPLES if e["direction"] == "easier"))
from google.colab import files
import json

# Upload the dataset
files.upload()  # upload difficulty_dataset_combined.json

# Load the dataset
with open("difficulty_dataset_combined.json") as f:
    train_data = json.load(f)

print(f"Loaded {len(train_data)} training examples")

# Build training examples from the dataset
NEW_EXAMPLES = []
for entry in train_data:
    base_question = entry.get("input", "")
    modified_question = entry.get("output", "")
    direction = entry.get("direction", "")

    if not base_question or not modified_question or not direction:
        continue

    NEW_EXAMPLES.append({
        "base_question": base_question,
        "modified_question": modified_question,
        "direction": direction,
    })

print(f"Valid training entries: {len(NEW_EXAMPLES)}")

from collections import Counter
directions = Counter(e["direction"] for e in NEW_EXAMPLES)
print("Directions:", dict(directions))
print("\nSample harder entry:")
print(next(e for e in NEW_EXAMPLES if e["direction"] == "harder"))
print("\nSample easier entry:")
print(next(e for e in NEW_EXAMPLES if e["direction"] == "easier"))

Saving difficulty_dataset_combined.json to difficulty_dataset_combined.json
Loaded 1374 training examples
Valid training entries: 1374
Directions: {'easier': 687, 'harder': 687}

Sample harder entry:
{'base_question': 'Explain the differences between Al-Idrisi’s map and the later European map of India.', 'modified_question': 'How did differences in knowledge, cartographic techniques, and historical context influence the contrasting representations of India in Al-Idrisi’s and later European maps?', 'direction': 'harder'}

Sample easier entry:
{'base_question': 'Explain the differences between Al-Idrisi’s map and the later European map of India.', 'modified_question': 'What is one difference between Al-Idrisi’s map and later maps?', 'direction': 'easier'}


Cell 5 — Tokenization

In [ ]:
from transformers import AutoTokenizer

BASE_MODEL = "google/gemma-2b"  # or your replacement model
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# def build_prompt_and_full_text(example):
#     direction = example["direction"]
#     instruction = example["instruction"]
#     question = example["base_question"]
#     answer = example["harder"] if direction == "harder" else example["easier"]

#     prompt = (
#         f"### System:\n{SYSTEM_PROMPT}\n\n"
#         f"### Input:\n"
#         f"Topic: {example['topic']}\n"
#         f"Original Question: {question}\n"
#         f"Target Difficulty: {direction}\n\n"
#         f"### Response:\n"
#     )
#     full_text = prompt + answer
#     return prompt, full_text


# def make_instruction_records(examples):
#     records = []
#     for ex in examples:
#         prompt, full_text = build_prompt_and_full_text(ex)
#         records.append({"prompt": prompt, "full_text": full_text})
#     return records


# def tokenize_with_masking(example):
#     prompt_ids = tokenizer(
#         example["prompt"],
#         add_special_tokens=False,
#         truncation=True,
#         max_length=MAX_LEN,
#     )["input_ids"]

#     encoded = tokenizer(
#         example["full_text"],
#         add_special_tokens=False,
#         truncation=True,
#         max_length=MAX_LEN,
#         padding="max_length",
#     )

#     labels = encoded["input_ids"].copy()
#     prompt_len = min(len(prompt_ids), MAX_LEN)
#     labels[:prompt_len] = [-100] * prompt_len
#     labels = [
#         t if m == 1 else -100
#         for t, m in zip(labels, encoded["attention_mask"])
#     ]
#     encoded["labels"] = labels
#     return encoded


# records = make_instruction_records(NEW_EXAMPLES)
# raw_ds = Dataset.from_list(records)

# tokenized_ds = raw_ds.map(
#     tokenize_with_masking,
#     remove_columns=raw_ds.column_names,
# )

# print(f"Tokenized dataset: {tokenized_ds}")
# print(f"Total records: {len(tokenized_ds)}")
# print(f"Example non-masked label tokens: "
#       f"{sum(1 for l in tokenized_ds[0]['labels'] if l != -100)}")
from datasets import Dataset

# System prompt for difficulty-controlled question generation
SYSTEM_PROMPT = """You are an expert at modifying questions to control their difficulty level.
Given a base question and a target difficulty (easier or harder), generate an appropriate modified version.
- For 'easier': Simplify the question, reduce complexity, make it more straightforward
- For 'harder': Add complexity, require deeper analysis, incorporate multiple concepts"""

MAX_LEN = 512  # Adjust based on your needs

def build_prompt_and_full_text(example):
    direction = example["direction"]
    base_question = example["base_question"]
    modified_question = example["modified_question"]

    prompt = (
        f"### System:\n{SYSTEM_PROMPT}\n\n"
        f"### Input:\n"
        f"Original Question: {base_question}\n"
        f"Target Difficulty: {direction}\n\n"
        f"### Response:\n"
    )
    full_text = prompt + modified_question
    return prompt, full_text


def make_instruction_records(examples):
    records = []
    for ex in examples:
        prompt, full_text = build_prompt_and_full_text(ex)
        records.append({"prompt": prompt, "full_text": full_text})
    return records


def tokenize_with_masking(example):
    prompt_ids = tokenizer(
        example["prompt"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )["input_ids"]

    encoded = tokenizer(
        example["full_text"],
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )

    labels = encoded["input_ids"].copy()
    prompt_len = min(len(prompt_ids), MAX_LEN)
    labels[:prompt_len] = [-100] * prompt_len
    labels = [
        t if m == 1 else -100
        for t, m in zip(labels, encoded["attention_mask"])
    ]
    encoded["labels"] = labels
    return encoded


records = make_instruction_records(NEW_EXAMPLES)
raw_ds = Dataset.from_list(records)

tokenized_ds = raw_ds.map(
    tokenize_with_masking,
    remove_columns=raw_ds.column_names,
)

print(f"Tokenized dataset: {tokenized_ds}")
print(f"Total records: {len(tokenized_ds)}")
print(f"Example non-masked label tokens: "
      f"{sum(1 for l in tokenized_ds[0]['labels'] if l != -100)}")

Map:   0%|          | 0/1374 [00:00<?, ? examples/s]

Tokenized dataset: Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 1374
})
Total records: 1374
Example non-masked label tokens: 124


Cell 7 — Add Stage 3 LoRA and trai

In [ ]:
!pip install -U bitsandbytes>=0.46.1 transformers peft accelerate


In [ ]:
!pip install -U "bitsandbytes>=0.46.1" "accelerate>=0.27.0" "transformers>=4.38.0" peft sentencepiece

In [ ]:
!pip install -U "bitsandbytes>=0.46.1" "accelerate>=0.27.0" "transformers>=4.38.0" peft sentencepiece


In [ ]:
print("Stage 1 exists:", STAGE1_DIR.exists(), list(STAGE1_DIR.iterdir()))
print("Stage 2 exists:", STAGE2_DIR.exists(), list(STAGE2_DIR.iterdir()))



Stage 1 exists: True [PosixPath('stage1_adapter/adapter_model.safetensors'), PosixPath('stage1_adapter/chat_template.jinja'), PosixPath('stage1_adapter/tokenizer.json'), PosixPath('stage1_adapter/adapter_config.json'), PosixPath('stage1_adapter/README.md'), PosixPath('stage1_adapter/tokenizer_config.json')]
Stage 2 exists: True [PosixPath('stage2_adapter/adapter_model.safetensors'), PosixPath('stage2_adapter/tokenizer.json'), PosixPath('stage2_adapter/adapter_config.json'), PosixPath('stage2_adapter/README.md'), PosixPath('stage2_adapter/tokenizer_config.json')]


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RUN_DIR = "/content/drive/MyDrive/stage3_runs"


Mounted at /content/drive


In [ ]:
# stage3_args = TrainingArguments(
#     output_dir=DRIVE_RUN_DIR,
#     per_device_train_batch_size=2,
#     gradient_accumulation_steps=4,
#     learning_rate=2e-4,
#     num_train_epochs=3,
#     lr_scheduler_type="cosine",
#     warmup_ratio=0.05,
#     fp16=True,
#     optim="paged_adamw_8bit",
#     logging_steps=20,
#     save_strategy="steps",
#     save_steps=200,          # adjust as needed
#     save_total_limit=2,      # keeps only recent checkpoints
#     report_to="none",
#     remove_unused_columns=False,
#     dataloader_pin_memory=False,
#     gradient_checkpointing=True,
#     max_grad_norm=0.3,
# )
from transformers import TrainingArguments

training_args = TrainingArguments(
  # or use local path if not using Drive
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=20,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training

In [ ]:
from transformers import Trainer, TrainingArguments

In [ ]:
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training

clear_memory()

# Load base model
base_model, tokenizer = load_base_model()

# ---- MERGE STAGE 1 ----
print("Loading + merging Stage 1...")
model = PeftModel.from_pretrained(base_model, str(STAGE1_DIR))
model = model.merge_and_unload()

# ---- MERGE STAGE 2 ----
print("Loading + merging Stage 2...")
model = PeftModel.from_pretrained(model, str(STAGE2_DIR))
model = model.merge_and_unload()

# ---- PREPARE FOR TRAINING ----
model = prepare_model_for_kbit_training(model)
model.gradient_checkpointing_enable()
model.config.use_cache = False

# ---- STAGE 3 LORA ----
lora3_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

stage3_model = get_peft_model(model, lora3_config)
stage3_model.print_trainable_parameters()
stage3_model.train()

# ---- TRAINING ARGS (NO DRIVE) ----
stage3_args = TrainingArguments(
    output_dir="/content/stage3_runs",   # local only
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,   # 🔥 reduced (more stable for stage 3)
    num_train_epochs=3,
    lr_scheduler_type="cosine",
    warmup_steps=50,
    fp16=True,
    optim="paged_adamw_8bit",
    logging_steps=20,
    save_strategy="no",   # 🔥 no checkpoint saving
    report_to="none",
    remove_unused_columns=False,
    dataloader_pin_memory=False,
    gradient_checkpointing=True,
    max_grad_norm=0.3,
)

# ---- TRAINER ----
trainer = Trainer(
    model=stage3_model,
    args=stage3_args,
    train_dataset=tokenized_ds,
)

# ---- TRAIN ----
print("Starting Stage 3 training...")
trainer.train()
print("Stage 3 training complete.")

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

Loading + merging Stage 1...


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:397: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


Loading + merging Stage 2...


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 1,843,200 || all params: 2,508,015,616 || trainable%: 0.0735
Starting Stage 3 training...


Step,Training Loss
20,8.672325
40,6.375782
60,3.703993
80,2.281542
100,2.042734
120,1.884770
140,1.815465
160,1.760221
180,1.707398
200,1.676750


Step,Training Loss
20,8.672325
40,6.375782
60,3.703993
80,2.281542
100,2.042734
120,1.884770
140,1.815465
160,1.760221
180,1.707398
200,1.676750


Stage 3 training complete.


In [ ]:
# Save locally inside Colab
final_save_path = "/content/final_model"
trainer.save_model(final_save_path)

print("Model saved locally. Zipping...")

# Zip the model folder
import shutil
shutil.make_archive("/content/final_model", 'zip', final_save_path)

print("Downloading...")

from google.colab import files
files.download("/content/final_model.zip")

Model saved locally. Zipping...
Downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# def generate_difficulty_controlled_question(base_question, target_difficulty):
#     """
#     Generate an easier or harder version of a question

#     Args:
#         base_question: The original question
#         target_difficulty: "easier" or "harder"
#     """
#     prompt = (
#         f"### System:\n{SYSTEM_PROMPT}\n\n"
#         f"### Input:\n"
#         f"Original Question: {base_question}\n"
#         f"Target Difficulty: {target_difficulty}\n\n"
#         f"### Response:\n"
#     )

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=200,
#             temperature=0.7,
#             top_p=0.9,
#             do_sample=True,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#     generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     # Extract just the response part
#     response = generated_text.split("### Response:\n")[-1].strip()

#     return response

# # Test the model
# test_question = "What is photosynthesis?"
# print(f"Base Question: {test_question}\n")

# easier = generate_difficulty_controlled_question(test_question, "easier")
# print(f"Easier Version: {easier}\n")

# harder = generate_difficulty_controlled_question(test_question, "harder")
# print(f"Harder Version: {harder}")

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Caching is incompatible with gradient checkpointing in GemmaDecoderLayer. Setting `past_key_values=None`.


Base Question: What is photosynthesis?

Easier Version: Explain Simplify things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things things thi

upload ur downloadded zip


In [ ]:
from google.colab import files
import zipfile
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# 🔥 STEP 1: Upload zip
uploaded = files.upload()

zip_name = list(uploaded.keys())[0]
extract_path = "/content/model_extracted"

# 🔥 STEP 2: Extract
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted to:", extract_path)
print("Files:", os.listdir(extract_path))

# 🔥 STEP 3: Load base model
BASE_MODEL = "google/gemma-2b"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    device_map="auto",
    torch_dtype=torch.float16
)

# 🔥 STEP 4: Load LoRA adapter
model = PeftModel.from_pretrained(base_model, extract_path)

model.eval()

print("✅ Model loaded successfully")

In [ ]:
# import torch

# def generate_rewrite(model, tokenizer, question, direction):
#     model.eval()

#     prompt = (
#         "### System:\n"
#         "You rewrite questions.\n"
#         "Rules:\n"
#         "- Output ONLY one question\n"
#         "- Do NOT explain\n"
#         "- Do NOT give answers\n"
#         "- Keep it concise\n"
#         "- Maintain same topic\n"
#         "- Match target difficulty strictly\n\n"

#         "### Input:\n"
#         f"Original Question: {question}\n"
#         f"Target Difficulty: {direction}\n\n"

#         "### Response:\n"
#     )

#     inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

#     with torch.no_grad():
#         outputs = model.generate(
#             **inputs,
#             max_new_tokens=32,          # 🔥 shorter = better control
#             do_sample=False,            # 🔥 deterministic
#             eos_token_id=tokenizer.eos_token_id,
#             pad_token_id=tokenizer.eos_token_id,
#         )

#     decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
#     result = decoded[len(prompt):].strip()

#     # 🔥 strong cleanup
#     result = result.split("\n")[0]              # remove extra lines
#     result = result.split("?")[0] + "?"        # force question ending

#     return result

FEW SHOT PROMPTS

In [ ]:
import torch

def generate_rewrite_fewshot(model, tokenizer, question, direction):
    model.eval()

    prompt = (
        "### System:\n"
        "You rewrite questions based on difficulty.\n"
        "Rules:\n"
        "- Output ONLY one question\n"
        "- No explanations\n"
        "- Keep same topic\n\n"

        "### Examples:\n"

        "Original: What is a cartographer?\n"
        "Easier: Who makes maps?\n"
        "Harder: Discuss the role of cartographers in shaping geographical knowledge.\n\n"

        "Original: Why do historians use sources?\n"
        "Easier: Why do historians use sources?\n"
        "Harder: Analyze how historical sources influence our understanding of the past.\n\n"

        "Original: What is a region?\n"
        "Easier: What is a region?\n"
        "Harder: Explain how regions develop distinct cultural identities.\n\n"

        "### Task:\n"
        f"Original: {question}\n"
        f"{direction.capitalize()}:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = decoded[len(prompt):].strip()

    result = result.split("\n")[0]
    result = result.split("?")[0] + "?"

    return result

In [ ]:
test_q = "what was the impact of akbar on mughal empire?"

print("EASIER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "easier"))

print("\nHARDER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "harder"))

EASIER:
How did Akbar promote trade?

HARDER:
Discuss the impact of Akbar’s administrative reforms on medieval India.?


In [ ]:
test_q = "What was the impact of Akbar on Mughal Empire?"

print("EASIER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "easier"))

print("\nHARDER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "harder"))

EASIER:
Who was Akbar?

HARDER:
Analyze the administrative reforms introduced by Akbar and how they changed Mughal Empire governance.?


In [ ]:
test_q = "How did trade influence medieval India?"

print("EASIER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "easier"))

print("\nHARDER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "harder"))

EASIER:
What were Indian trading towns?

HARDER:
Analyze how trade helped people in medieval India.?


In [ ]:

test_q = "What role did religion play in politics?"

print("EASIER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "easier"))

print("\nHARDER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "harder"))

EASIER:
How did religious practices change over time?

HARDER:
Discuss how religious beliefs influenced political structures.?


In [ ]:

test_q = "How did migration affect society?"

print("EASIER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "easier"))

print("\nHARDER:")
print(generate_rewrite_fewshot(stage3_model, tokenizer, test_q, "harder"))

EASIER:
What were some early medieval Indian towns?

HARDER:
Discuss how migration changed social life in medieval India.?


In [ ]:
# 20 HARD GENERALIZATION TEST QUESTIONS
test_questions_hard = [
    "What was the impact of Akbar on the Mughal Empire?",
    "How did trade influence medieval India?",
    "Why did agriculture expand during this period?",
    "How did migration affect society?",
    "What role did religion play in politics?",
    "Why did empires rise and fall?",
    "How did technological innovations change medieval society?",
    "What factors led to the growth of towns in medieval India?",
    "How did the Bhakti movement influence social structures?",
    "What was the role of markets in economic development?",
    "How did regional identities develop in medieval India?",
    "What impact did new crops have on agriculture and society?",
    "How did rulers maintain control over large empires?",
    "What were the causes of economic inequality among peasants?",
    "How did trade networks contribute to cultural exchange?",
    "What role did language play in shaping cultural identity?",
    "How did religious beliefs influence everyday life?",
    "What factors contributed to the decline of empires?",
    "How did governance systems maintain political stability?",
    "What was the significance of temples in medieval society?",
]

# RUN TESTS
def run_tests(model, tokenizer, questions):
    for i, q in enumerate(questions, 1):
        print("="*60)
        print(f"Q{i}: {q}")

        easier = generate_rewrite_fewshot(model, tokenizer, q, "easier")
        harder = generate_rewrite_fewshot(model, tokenizer, q, "harder")

        print("EASIER:", easier)
        print("HARDER:", harder)
        print()

run_tests(stage3_model, tokenizer, test_questions_hard)

Q1: What was the impact of Akbar on the Mughal Empire?
EASIER: Analyze the administrative reforms introduced by Akbar and how they changed Mughal Empire administration.?
HARDER: Analyze the administrative reforms introduced by Akbar and how they changed Mughal administrative structure.?

Q2: How did trade influence medieval India?
EASIER: What were Indian trading towns?
HARDER: Analyze how trade helped people in medieval India.?

Q3: Why did agriculture expand during this period?
EASIER: Discuss how economic activities changed in medieval India.?
HARDER: Discuss how economic activities changed in medieval India.?

Q4: How did migration affect society?
EASIER: What were some early medieval Indian towns?
HARDER: Discuss how migration changed social life in medieval India.?

Q5: What role did religion play in politics?
EASIER: How did religious practices change over time?
HARDER: Discuss how religious beliefs influenced political structures.?

Q6: Why did empires rise and fall?
EASIER: Wh

In [ ]:
q = "Explain how agriculture affected society in medieval India."

print("EASIER:", generate_rewrite_fewshot(stage3_model, tokenizer, q, "easier"))
print("HARDER:", generate_rewrite_fewshot(stage3_model, tokenizer, q, "harder"))

EASIER: What was medieval Indian agriculture?
HARDER: Discuss the economic importance of agriculture in medieval India.?


In [ ]:
STRONG_FEWSHOT_PROMPT = """
You rewrite questions based on difficulty.

Rules:
- Output ONLY one question
- No explanations
- Easier = shorter, simpler words, direct
- Harder = analytical, include causes, impact, or reasoning

Examples:

Original: What was the impact of Akbar on the Mughal Empire?
Easier: What did Akbar do for the Mughal Empire?
Harder: Analyze the political and cultural impact of Akbar on the Mughal Empire.

Original: How did trade influence medieval India?
Easier: How did trade help people?
Harder: Explain how trade influenced economic and cultural development in medieval India.

Original: Why did agriculture expand during this period?
Easier: Why did farming increase?
Harder: Analyze the economic and environmental factors behind agricultural expansion.

Original: How did migration affect society?
Easier: Why did people move?
Harder: Discuss the social and economic impact of migration on medieval society.

Original: What role did religion play in politics?
Easier: How did religion affect rulers?
Harder: Analyze the role of religion in shaping political authority and governance.

Original: Why did empires rise and fall?
Easier: Why did empires grow and decline?
Harder: Evaluate the factors responsible for the rise and fall of empires.

Original: What is a cartographer?
Easier: Who makes maps?
Harder: Discuss the role of cartographers in shaping geographical knowledge.

Original: What are manuscripts?
Easier: What are handwritten books?
Harder: Explain the importance of manuscripts in preserving historical records.

Original: What is a region?
Easier: What is a region?
Harder: Explain how regions develop distinct cultural and political identities.
"""

In [ ]:
import torch

def generate_rewrite_strong(model, tokenizer, question, direction):
    model.eval()

    prompt = (
        "### Instruction:\n"
        + STRONG_FEWSHOT_PROMPT +
        "\n### Task:\n"
        f"Original: {question}\n"
        f"{direction.capitalize()}:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = decoded[len(prompt):].strip()

    # cleanup
    result = result.split("\n")[0]
    result = result.split("?")[0] + "?"

    return result

In [ ]:
q = "What was the impact of Akbar on the Mughal Empire?"

print("EASIER:", generate_rewrite_strong(stage3_model, tokenizer, q, "easier"))
print("HARDER:", generate_rewrite_strong(stage3_model, tokenizer, q, "harder"))

EASIER: What was Akbar?
HARDER: Analyze the administrative and cultural impact of Akbar’s administrative reforms in India.?


In [ ]:
# 20 HARD GENERALIZATION TEST QUESTIONS
test_questions_hard = [
    "What was the impact of Akbar on the Mughal Empire?",
    "How did trade influence medieval India?",
    "Why did agriculture expand during this period?",
    "How did migration affect society?",
    "What role did religion play in politics?",
    "Why did empires rise and fall?",
    "How did technological innovations change medieval society?",
    "What factors led to the growth of towns in medieval India?",
    "How did the Bhakti movement influence social structures?",
    "What was the role of markets in economic development?",
    "How did regional identities develop in medieval India?",
    "What impact did new crops have on agriculture and society?",
    "How did rulers maintain control over large empires?",
    "What were the causes of economic inequality among peasants?",
    "How did trade networks contribute to cultural exchange?",
    "What role did language play in shaping cultural identity?",
    "How did religious beliefs influence everyday life?",
    "What factors contributed to the decline of empires?",
    "How did governance systems maintain political stability?",
    "What was the significance of temples in medieval society?",
]

# RUN TESTS
def run_tests(model, tokenizer, questions):
    for i, q in enumerate(questions, 1):
        print("="*60)
        print(f"Q{i}: {q}")

        easier = generate_rewrite_strong(model, tokenizer, q, "easier")
        harder = generate_rewrite_strong(model, tokenizer, q, "harder")

        print("EASIER:", easier)
        print("HARDER:", harder)
        print()

run_tests(stage3_model, tokenizer, test_questions_hard)

Q1: What was the impact of Akbar on the Mughal Empire?
EASIER: What was Akbar?
HARDER: Analyze the administrative and cultural impact of Akbar’s administrative reforms in India.?

Q2: How did trade influence medieval India?
EASIER: What was trade?
HARDER: Discuss how trade helped people in medieval India.?

Q3: Why did agriculture expand during this period?
EASIER: What was agrarian expansion?
HARDER: Discuss the economic and social factors behind agricultural expansion.?

Q4: How did migration affect society?
EASIER: What is migration?
HARDER: Discuss the social and economic impact of migration on medieval society.?

Q5: What role did religion play in politics?
EASIER: How did religious beliefs influence political authority?
HARDER: Discuss the ways in which religion shaped political authority in medieval India.?

Q6: Why did empires rise and fall?
EASIER: What were some early medieval Indian empires?
HARDER: Discuss the factors that led to the rise and fall of empires.?

Q7: How did 

In [ ]:
STRONG_FEWSHOT_PROMPT_V2 = """
You rewrite questions based on difficulty.

Rules:
- Output ONLY one question
- Easier = simpler words, shorter
- Harder = deeper thinking, NOT just "analyze" or "discuss"
- Use VARIED styles:
  - Why / How / To what extent / In what ways / What factors

Examples:

Original: What was the impact of Akbar on the Mughal Empire?
Easier: What did Akbar do for the Mughal Empire?
Harder: In what ways did Akbar influence the political and cultural structure of the Mughal Empire?

Original: How did trade influence medieval India?
Easier: How did trade help people?
Harder: What factors made trade important for economic and cultural development in medieval India?

Original: Why did agriculture expand?
Easier: Why did farming increase?
Harder: What conditions led to the expansion of agriculture during this period?

Original: How did migration affect society?
Easier: Why did people move?
Harder: How did migration change the social and economic structure of society?

Original: Why did empires rise and fall?
Easier: Why did empires grow and decline?
Harder: What were the main reasons behind the rise and fall of empires?

Original: What role did religion play in politics?
Easier: How did religion affect rulers?
Harder: How did religious beliefs shape political power and authority?
"""

In [ ]:
import torch

# 🔥 STRONG FEWSHOT WITH VARIETY
STRONG_FEWSHOT_PROMPT_V3 = """
Rewrite the question based on difficulty.

Rules:
- Output ONLY one question
- Easier = simpler, shorter
- Harder = deeper thinking (causes, impact, reasoning)
- MUST stay on the SAME topic

Examples:

Original: What was the impact of Akbar on the Mughal Empire?
Easier: What did Akbar do for the Mughal Empire?
Harder: What effects did Akbar have on the Mughal Empire?

Original: How did trade influence medieval India?
Easier: How did trade help people?
Harder: What factors made trade important in medieval India?
"""

# 🔥 GENERATE FUNCTION WITH BANNED WORDS
def generate_rewrite_strong_v3(model, tokenizer, question, direction):
    model.eval()

    prompt = (
        STRONG_FEWSHOT_PROMPT_V3 +
        "\nTask:\n"
        f"Original: {question}\n"
        f"{direction.capitalize()}:"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # 🔥 BAN OVERUSED WORDS
    bad_words = ["Analyze", "Discuss"]
    bad_words_ids = [
        tokenizer(word, add_special_tokens=False).input_ids
        for word in bad_words
    ]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            bad_words_ids=bad_words_ids,   # 🔥 key fix
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
    result = decoded[len(prompt):].strip()

    # 🔥 CLEAN OUTPUT
    result = result.split("\n")[0]

    # remove leftover patterns if any
    for w in ["Analyze", "Discuss"]:
        result = result.replace(w, "").strip()

    # ensure question format
    if "?" in result:
        result = result.split("?")[0] + "?"
    else:
        result = result + "?"

    return result


# 🔥 QUICK TEST
q = "What was the impact of Akbar on the Mughal Empire?"

print("EASIER:", generate_rewrite_strong_v3(stage3_model, tokenizer, q, "easier"))
print("HARDER:", generate_rewrite_strong_v3(stage3_model, tokenizer, q, "harder"))

EASIER: What was Akbar?
HARDER: What were the administrative reforms of Akbar?


In [ ]:
# 20 HARD GENERALIZATION TEST QUESTIONS
test_questions_hard = [
    "What was the impact of Akbar on the Mughal Empire?",
    "How did trade influence medieval India?",
    "Why did agriculture expand during this period?",
    "How did migration affect society?",
    "What role did religion play in politics?",
    "Why did empires rise and fall?",
    "How did technological innovations change medieval society?",
    "What factors led to the growth of towns in medieval India?",
    "How did the Bhakti movement influence social structures?",
    "What was the role of markets in economic development?",
    "How did regional identities develop in medieval India?",
    "What impact did new crops have on agriculture and society?",
    "How did rulers maintain control over large empires?",
    "What were the causes of economic inequality among peasants?",
    "How did trade networks contribute to cultural exchange?",
    "What role did language play in shaping cultural identity?",
    "How did religious beliefs influence everyday life?",
    "What factors contributed to the decline of empires?",
    "How did governance systems maintain political stability?",
    "What was the significance of temples in medieval society?",
]

# RUN TESTS
def run_tests(model, tokenizer, questions):
    for i, q in enumerate(questions, 1):
        print("="*60)
        print(f"Q{i}: {q}")

        easier = generate_rewrite_strong_v3(model, tokenizer, q, "easier")
        harder = generate_rewrite_strong_v3(model, tokenizer, q, "harder")

        print("EASIER:", easier)
        print("HARDER:", harder)
        print()

run_tests(stage3_model, tokenizer, test_questions_hard)

Q1: What was the impact of Akbar on the Mughal Empire?
EASIER: What was Akbar?
HARDER: What were the administrative reforms of Akbar?

Q2: How did trade influence medieval India?
EASIER: What were Indian trading towns?
HARDER: What factors made trade important in medieval India?

Q3: Why did agriculture expand during this period?
EASIER: What were the major crops?
HARDER: What factors contributed to the expansion of agriculture?

Q4: How did migration affect society?
EASIER: What were some medieval Indian towns?
HARDER: What were some major medieval Indian trading towns?

Q5: What role did religion play in politics?
EASIER: What was a religion?
HARDER: How did religion influence political authority in medieval India?

Q6: Why did empires rise and fall?
EASIER: What were empires?
HARDER: What were some of the major factors?

Q7: How did technological innovations change medieval society?
EASIER: What were some medieval Indian towns?
HARDER: What were some medieval Indian towns?

Q8: What